# Where the smoothness breaks

An integrator assumes the thing it is integrating is smooth across the step it is taking.
Two things in this toolchain break that assumption on purpose: a **state event**, where a
`when` condition crosses zero and the model changes discontinuously, and a **state-set
switch**, where the set of variables being integrated is swapped out mid-run because the
one in use went singular.

Both stop the solver, change something, and start it again. This notebook looks at what
that does to a trajectory — and at the numbers that say the engine did it right.

```bash
pip install "modelica[sim]" matplotlib ipywidgets
jupyter lab docs/notebooks/events.ipynb
```

In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples" / "models").is_dir())

SURFACE, INK, MUTED, LINE = "#fcfcfb", "#0b0b0b", "#52514e", "#d4d3cd"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"

mpl.rcParams.update(
    {
        "figure.figsize": (8.4, 3.4),
        "figure.dpi": 110,
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "axes.edgecolor": LINE,
        "axes.labelcolor": MUTED,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.titlelocation": "left",
        "axes.titlepad": 10,
        "axes.titlesize": 11,
        "grid.color": "#e8e7e2",
        "grid.linewidth": 0.8,
        "lines.linewidth": 2.0,
        "legend.frameon": False,
        "text.color": INK,
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "font.size": 10,
    }
)
%config InlineBackend.figure_formats = ["svg"]

## 1. One event per bounce

`BouncingBall` is two equations and a `when`: `der(h) = v`, `der(v) = -g`, and
`reinit(v, -e * pre(v))` when the height reaches zero. Every bounce is a state event, and
the run reports them with the Modelica condition that fired.

In [ ]:
from dataclasses import replace

from modelica.ir import nodes as ir
from modelica.lang import parse
from modelica.sim import causalize, flatten, simulate

ball_flat = flatten(
    [parse((root / "examples" / "models" / "BouncingBall.mo").read_text())], "BouncingBall"
)
ball = simulate(causalize(ball_flat), stop=4.0, points=1200)

print(ball)
for event in ball.events[:4]:
    print(f"  {event}")
print(f"  ... {len(ball.events)} in total")

In [ ]:
fig, ax = plt.subplots()
ax.plot(ball.time, ball["h"], color=BLUE)
for event in ball.events:
    ax.axvline(event.time, color=LINE, linewidth=1, zorder=0)
ax.axhline(0, color=MUTED, linewidth=1)
ax.set_title(f"Height, and the {len(ball.events)} events that interrupted the integration")
ax.set_xlabel("time  [s]")
ax.set_ylabel("h  [m]")
ax.set_ylim(-0.02, 1.05)
plt.show()

## 2. What the restart costs

Asked for no output grid, `simulate` returns the solver's own steps — the honest picture of
where it had to work. Each bounce shows up twice: as a step cut short to land exactly on the
crossing, and as a fresh start on the other side of it.

In [ ]:
raw = simulate(causalize(ball_flat), stop=4.0)
gap = np.diff(raw.time)
moving = gap > 0

fig, ax = plt.subplots()
ax.semilogy(raw.time[:-1][moving], gap[moving], color=BLUE, marker="o", markersize=4, linewidth=1.2)
for event in ball.events:
    ax.axvline(event.time, color=LINE, linewidth=1, zorder=0)
ax.set_title(
    f"Step size taken, {len(raw.time)} steps — and {int((~moving).sum())} restarts of zero length"
)
ax.set_xlabel("time  [s]")
ax.set_ylabel("Δt  [s]")
plt.show()

The steps collapse into each bounce and recover after it. The zero-length entries are the
restarts themselves: the solver stops *at* the crossing and the next segment begins at the
same instant with a different velocity, so the trajectory has two values at one time. That
is not a numerical artifact to be smoothed over — it is what a discontinuity is.

## 3. The bounces are a geometric series, and the run says so

With restitution `e`, each bounce keeps a fraction `e` of the impact speed, so apex heights
fall by `e²` and the intervals between bounces by `e`. Both are checkable against what came
out of the integrator.

In [ ]:
# Complete flights only: from the start (or a bounce) to the next bounce. The stretch after
# the last event is still in the air when the run ends, so its apex is not one.
from itertools import pairwise

AIRBORNE = 1e-9  # below this the ball is resting, not flying, and its "apex" is arithmetic noise

apex = []
for start, end in pairwise([0.0, *(event.time for event in ball.events)]):
    window = (ball.time >= start) & (ball.time <= end)
    if window.any():
        apex.append(float(ball["h"][window].max()))

e = 0.8
apex = np.array([h for h in apex if h > AIRBORNE])
n = np.arange(len(apex))
predicted = apex[0] * e ** (2 * n)

fig, (left, right) = plt.subplots(1, 2, figsize=(8.4, 3.2))
left.semilogy(n, predicted, color=ORANGE, linewidth=1.6, linestyle=(0, (4, 3)))
left.semilogy(n, apex, color=BLUE, marker="o", markersize=6, linewidth=0)
left.annotate(
    "h₀·e²ⁿ",
    xy=(n[-3], predicted[-3]),
    xytext=(-38, -6),
    textcoords="offset points",
    color=ORANGE,
    fontsize=9,
)
left.annotate(
    "measured apex",
    xy=(n[4], apex[4]),
    xytext=(10, 10),
    textcoords="offset points",
    color=BLUE,
    fontsize=9,
)
left.set_title("Apex height per flight")
left.set_xlabel("flight")
left.set_ylabel("apex  [m]")

intervals = np.diff([event.time for event in ball.events])
ratio = intervals[1:] / intervals[:-1]
right.plot(np.arange(1, len(ratio) + 1), ratio, color=BLUE, marker="o", markersize=6)
right.axhline(e, color=ORANGE, linewidth=1.6, linestyle=(0, (4, 3)))
right.annotate(
    f"e = {e}", xy=(1, e), xytext=(0, 10), textcoords="offset points", color=ORANGE, fontsize=9
)
right.set_title("Ratio of successive flight times")
right.set_xlabel("bounce")
right.set_ylabel("tₙ₊₁ / tₙ")
right.set_ylim(0.74, 0.86)
plt.tight_layout()
plt.show()

print(f"mean interval ratio {ratio.mean():.9f}  against e = {e}")
print(f"worst apex error   {np.abs(np.array(apex) - predicted).max():.2e} m")

The interval ratio is the event locator being graded: it comes from the times the solver
reported for the crossings, not from anything analytic. Nine digits of agreement means the
zero-crossing was found to solver tolerance every time, not merely bracketed.

## 4. Restitution on a slider

Dropping `e` makes the bounces die out sooner; raising it toward 1 pushes the run into the
Zeno regime, where infinitely many events pile up before a finite time. The engine gives up
after a hundred consecutive zero-length segments and says so rather than hanging.

In [ ]:
def bounce(e: float, h0: float, stop: float) -> None:
    """Simulate the ball with that restitution and drop height, and plot it."""
    values = {"e": e}
    starts = {"h": h0}
    tuned = replace(
        ball_flat,
        variables=tuple(
            replace(
                v,
                binding=ir.RealLiteral(values[v.name]) if v.name in values else v.binding,
                start=ir.RealLiteral(starts[v.name]) if v.name in starts else v.start,
            )
            for v in ball_flat.variables
        ),
    )
    result = simulate(causalize(tuned), stop=stop, points=1200)

    fig, ax = plt.subplots()
    ax.plot(result.time, result["h"], color=BLUE)
    for event in result.events:
        ax.axvline(event.time, color=LINE, linewidth=1, zorder=0)
    ax.axhline(0, color=MUTED, linewidth=1)
    ax.set_title(
        f"e = {e:.2f}, dropped from {h0:.2f} m — {len(result.events)} bounces in {stop:.0f} s"
    )
    ax.set_xlabel("time  [s]")
    ax.set_ylabel("h  [m]")
    ax.set_ylim(-0.02, h0 * 1.06)
    plt.show()


bounce(e=0.8, h0=1.0, stop=4.0)

In [ ]:
import ipywidgets as widgets

widgets.interact(
    bounce,
    e=widgets.FloatSlider(
        value=0.8, min=0.1, max=0.95, step=0.05, description="e", continuous_update=False
    ),
    h0=widgets.FloatSlider(
        value=1.0, min=0.2, max=3.0, step=0.1, description="h start", continuous_update=False
    ),
    stop=widgets.FloatSlider(
        value=4.0, min=1.0, max=12.0, step=1.0, description="stop", continuous_update=False
    ),
)

## 5. The other kind of interruption

The pendulum has no `when` in it and still gets interrupted. `x² + y² = L²` makes it an
index-3 DAE, and index reduction leaves two admissible state sets — `(y, vy)` and
`(x, vx)` — neither of which works all the way around the swing. The engine switches
between them while the run is going, and each switch is an event with the set it moved to
as its cause.

In [ ]:
pendulum_flat = flatten(
    [parse((root / "examples" / "models" / "Pendulum.mo").read_text())], "Pendulum"
)
pendulum = causalize(pendulum_flat)
swing = simulate(pendulum, stop=4.0, points=1600)

print(f"state sets: {pendulum.candidates}")
for event in swing.events:
    print(f"  {event}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(swing.time, swing["x"], color=BLUE)
ax.plot(swing.time, swing["y"], color=ORANGE)
ax.annotate(
    "x",
    xy=(swing.time[-1], swing["x"][-1]),
    xytext=(6, 0),
    textcoords="offset points",
    color=BLUE,
    fontsize=10,
    va="center",
)
ax.annotate(
    "y",
    xy=(swing.time[-1], swing["y"][-1]),
    xytext=(6, 0),
    textcoords="offset points",
    color=ORANGE,
    fontsize=10,
    va="center",
)
for event in swing.events:
    ax.axvline(event.time, color=LINE, linewidth=1, zorder=0)
ax.set_title(f"The swing, with the {len(swing.events)} state-set switches marked")
ax.set_xlabel("time  [s]")
ax.set_ylabel("position  [m]")
plt.show()

## 6. Why it has to switch

Each state set causalizes to its own blocks, and each carries a **pivot** — the expression
the solve divides by, kept symbolically so it can be evaluated along the trajectory. For
`(y, vy)` it is `4x²`; for `(x, vx)` it is `4y²`. Where a pivot passes through zero its set
is unusable, and structural analysis cannot see that coming: it is a statement about
numbers, not about incidence.

Evaluating both along the run puts the whole argument on one chart. They cross where
`|x| = |y|`, which is 45 degrees, which is the answer the literature gives.

In [ ]:
from modelica.ir.evaluate import evaluate
from modelica.lang import unparse_expr

pivots = []
for mode in pendulum.modes:
    print(f"{mode.states}  pivot = {unparse_expr(mode.pivot)}")
    pivots.append(
        np.array([abs(float(evaluate(mode.pivot, swing.at(float(t))))) for t in swing.time])
    )

fig, ax = plt.subplots()
for series, mode, colour in zip(pivots, pendulum.modes, (BLUE, ORANGE), strict=True):
    ax.semilogy(
        swing.time, np.maximum(series, 1e-12), color=colour, label=f"({', '.join(mode.states)})"
    )
for event in swing.events:
    ax.axvline(event.time, color=LINE, linewidth=1, zorder=0)
ax.set_title("How well conditioned each state set is, along the trajectory")
ax.set_xlabel("time  [s]")
ax.set_ylabel("|pivot|")
ax.legend(loc="lower left", ncols=2, labelcolor=MUTED)
plt.show()

The switches land near the crossings, which is the engine choosing the better-conditioned
set — not exactly on them, because a set is only abandoned once it actually stops working,
and "stops working" is something the linear algebra reports rather than something a
threshold predicts.

## 7. The point of all of it

A constraint that has been differentiated twice is no longer in the equations being
integrated. Nothing in the right-hand side says the rod has a fixed length any more — so if
index reduction and state selection were subtly wrong, the mass would drift off the circle
while satisfying every equation the solver was given.

It does not:

In [ ]:
EPS = np.finfo(float).eps

drift = np.abs(np.hypot(swing["x"], swing["y"]) - 1.0)
worst = np.maximum.accumulate(drift) / EPS  # worst so far, in machine epsilons

fig, ax = plt.subplots()
ax.step(swing.time, worst, where="post", color=BLUE)
ax.axhline(1.0, color=ORANGE, linewidth=1.6, linestyle=(0, (4, 3)))
ax.annotate(
    "one machine epsilon",
    xy=(0.15, 1.0),
    xytext=(0, 8),
    textcoords="offset points",
    color=ORANGE,
    fontsize=9,
)
for event in swing.events:
    ax.axvline(event.time, color=LINE, linewidth=1, zorder=0)
ax.set_title("Worst drift off the rod so far, over the whole run")
ax.set_ylim(0, 1.6)
ax.set_xlabel("time  [s]")
ax.set_ylabel("| √(x²+y²) − L |  [ε]")
plt.show()

print(f"worst drift over the run: {drift.max():.3e} m = {drift.max() / EPS:.2f} machine epsilons")

## 8. Amplitude on a slider

The number of switches is not a property of the model, it is a property of the run. A small
swing stays where `(y, vy)` is fine and never switches at all; a large one crosses 45
degrees four times per period.

In [ ]:
def swing_from(degrees: float, stop: float) -> None:
    """Release the pendulum from that angle off vertical and count the switches."""
    theta = np.radians(degrees)
    starts = {"x": float(np.sin(theta)), "y": float(-np.cos(theta))}
    tuned = replace(
        pendulum_flat,
        variables=tuple(
            replace(v, start=ir.RealLiteral(starts[v.name]) if v.name in starts else v.start)
            for v in pendulum_flat.variables
        ),
    )
    result = simulate(causalize(tuned), stop=stop, points=1200)

    fig, (left, right) = plt.subplots(1, 2, figsize=(8.4, 3.2))
    left.plot(result.time, result["x"], color=BLUE)
    left.plot(result.time, result["y"], color=ORANGE)
    for event in result.events:
        left.axvline(event.time, color=LINE, linewidth=1, zorder=0)
    left.set_title(f"{degrees:.0f}° off vertical — {len(result.events)} switches")
    left.set_xlabel("time  [s]")
    left.set_ylabel("position  [m]")
    left.set_ylim(-1.15, 1.15)

    right.plot(result["x"], result["y"], color=BLUE, linewidth=1.4)
    right.plot([0.0], [0.0], marker="o", markersize=8, color=MUTED)
    right.set_title("Path of the mass")
    right.set_xlabel("x  [m]")
    right.set_ylabel("y  [m]")
    right.set_xlim(-1.15, 1.15)
    right.set_ylim(-1.15, 1.15)
    right.set_aspect("equal")
    plt.tight_layout()
    plt.show()


swing_from(degrees=90.0, stop=4.0)

In [ ]:
widgets.interact(
    swing_from,
    degrees=widgets.FloatSlider(
        value=90.0, min=5.0, max=175.0, step=5.0, description="angle °", continuous_update=False
    ),
    stop=widgets.FloatSlider(
        value=4.0, min=1.0, max=10.0, step=1.0, description="stop", continuous_update=False
    ),
)

Both interruptions come back to the same design point: the discrete half of a simulation is
*finite*, so it can be enumerated rather than sampled. Which is
[the next notebook](../verifying/) — the same switching logic, handed to a model checker
instead of an integrator.